<a href="https://colab.research.google.com/github/parisazeynaly/Advanced--Statistics-Emprical/blob/main/RL_Exploration_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**CORE Q-LEARNING LOOP ON FROZENLAKE**

In [ ]:
import numpy as np
import gymnasium as gym


from scipy import stats
from itertools import product
import json
import os
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ── Try to import plotting
try:
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches
    PLOT = True
except ImportError:
    PLOT = False
    print("matplotlib not found — skipping plots. Install with: pip install matplotlib")

os.makedirs("results", exist_ok=True)

In [ ]:
# HYPERPARAMETERS
ALPHA       = 0.1      # learning rate — how much each update shifts Q
GAMMA       = 0.99     # discount factor — how much future reward matters
EPISODES    = 5000      # total training episodes
EPS_START   = 1.0      # initial exploration rate (100% random at start)
EPS_END     = 0.01     # final exploration rate (1% random at the end)
MAP_NAME    = "4x4"    # FrozenLake grid size: "4x4" or "8x8"
IS_SLIPPERY = False    # False = deterministic, True = stochastic (slips)
SEED        = 42       # for reproducibility

EPSILON DECAY

In [ ]:
def get_epsilon(episode, total_episodes):
    """
    Linearly decay epsilon from EPS_START to EPS_END over training.

    Why decay? Early on, the agent knows nothing, so it should explore
    a lot (high epsilon). As training progresses, the Q-table becomes
    more reliable, so the agent should exploit what it has learned
    (low epsilon).
    """
    fraction = episode / total_episodes
    epsilon = EPS_START - fraction * (EPS_START - EPS_END)
    return max(EPS_END, epsilon)

ACTION SELECTION (epsilon-greedy — the simplest strategy)

In [ ]:
def choose_action(Q, state, epsilon, n_actions, rng):
    """
    With probability epsilon: pick a uniformly random action (explore).
    With probability 1-epsilon: pick the action with the highest known
    Q-value for this state (exploit).
    """
    if rng.random() < epsilon:
        return rng.integers(n_actions)          # explore
    else:
        return int(np.argmax(Q[state]))         # exploit

THE Q-LEARNING UPDATE (the Bellman equation )

In [ ]:
def update_q(Q, state, action, reward, next_state, terminated):
    """
    Q(s,a) <- Q(s,a) + alpha * [ r + gamma * max_a' Q(s',a') - Q(s,a) ]

    Read it left to right:
      - "r + gamma * max_a' Q(s',a')"  is the TD TARGET — our new best
        guess of how good (s,a) really is, using the reward we just got
        plus the best value we can get from the next state.
      - "... - Q(s,a)"  is how far off our OLD estimate was. This
        difference is called the TD ERROR.
      - We nudge Q(s,a) toward the target by a small step (alpha).

    If next_state is terminal (hole or goal), there is no future —
    so the "max_a' Q(s',a')" term is zero.
    """
    if terminated:
        td_target = reward
    else:
        td_target = reward + GAMMA * np.max(Q[next_state])

    td_error = td_target - Q[state, action]
    Q[state, action] += ALPHA * td_error

    return td_error   # returned only so we can print/inspect it while learning

In [ ]:
def run_episode(env, Q, epsilon, rng):
    """
    Plays one full episode: from reset() until the agent reaches the
    goal, falls in a hole, or the episode times out.

    Returns the total reward collected (0 or 1 for FrozenLake, since
    reward is sparse — only the goal gives +1).
    """
    state, _ = env.reset(seed=int(rng.integers(1_000_000)))
    n_actions = env.action_space.n
    total_reward = 0.0

    while True:
        action = choose_action(Q, state, epsilon, n_actions, rng)
        next_state, reward, terminated, truncated, _ = env.step(action)

        update_q(Q, state, action, reward, next_state, terminated)

        total_reward += reward
        state = next_state

        if terminated or truncated:
            break

    return total_reward

In [ ]:
def train():
    rng = np.random.default_rng(SEED)
    env = gym.make("FrozenLake-v1", map_name=MAP_NAME, is_slippery=IS_SLIPPERY)

    n_states  = env.observation_space.n
    n_actions = env.action_space.n
    Q = np.zeros((n_states, n_actions))

    rewards_per_episode = []

    for ep in range(EPISODES):
        epsilon = get_epsilon(ep, EPISODES)
        reward = run_episode(env, Q, epsilon, rng)
        rewards_per_episode.append(reward)

        # Print progress every 500 episodes so you can watch it learn
        if (ep + 1) % 500 == 0:
            recent_success_rate = np.mean(rewards_per_episode[-500:])
            print(f"Episode {ep+1:5d} | epsilon={epsilon:.3f} | "
                  f"success rate (last 500 eps) = {recent_success_rate:.2%}")

    env.close()
    return Q, rewards_per_episode

SANITY CHECKS

In [ ]:
def evaluate(Q, n_eval_episodes=100):
    """
    Run the trained agent with epsilon=0 (pure exploitation, no randomness)
    to see its true success rate. This is what "performance" means.
    """
    env = gym.make("FrozenLake-v1", map_name=MAP_NAME, is_slippery=IS_SLIPPERY)
    rng = np.random.default_rng(123)  # different seed than training
    successes = 0

    for _ in range(n_eval_episodes):
        state, _ = env.reset(seed=int(rng.integers(1_000_000)))
        for _ in range(200):  # safety cap on steps per episode
            action = int(np.argmax(Q[state]))   # always greedy, no exploration
            state, reward, terminated, truncated, _ = env.step(action)
            if terminated or truncated:
                successes += reward
                break

    env.close()
    return successes / n_eval_episodes


def print_q_table(Q):
    """Pretty-print the Q-table so you can inspect it visually."""
    action_names = ["LEFT", "DOWN", "RIGHT", "UP"]
    print("\nFinal Q-table:")
    print("State | " + " | ".join(f"{a:>7s}" for a in action_names))
    for s in range(Q.shape[0]):
        row = " | ".join(f"{Q[s, a]:7.3f}" for a in range(Q.shape[1]))
        print(f"{s:5d} | {row}")


In [ ]:
if __name__ == "__main__":
    print(f"Training Q-learning on FrozenLake ({MAP_NAME}, "
          f"slippery={IS_SLIPPERY}) for {EPISODES} episodes...\n")

    Q, rewards = train()

    print_q_table(Q)

    success_rate = evaluate(Q)
    print(f"\nEvaluation: agent reaches the goal {success_rate:.1%} of the time "
          f"over 100 greedy (no-exploration) episodes.")

    if success_rate > 0.7:
        print("✓ Looks correct — the agent has clearly learned a good policy.")
    else:
        print("⚠ Success rate is low. Possible causes: too few episodes, "
              "epsilon not decaying enough, or a bug in the update rule. "
              "Discuss with your teammate before moving to Phase 2.")

Training Q-learning on FrozenLake (4x4, slippery=False) for 5000 episodes...

Episode   500 | epsilon=0.901 | success rate (last 500 eps) = 2.40%
Episode  1000 | epsilon=0.802 | success rate (last 500 eps) = 5.80%
Episode  1500 | epsilon=0.703 | success rate (last 500 eps) = 13.00%
Episode  2000 | epsilon=0.604 | success rate (last 500 eps) = 20.00%
Episode  2500 | epsilon=0.505 | success rate (last 500 eps) = 31.20%
Episode  3000 | epsilon=0.406 | success rate (last 500 eps) = 43.00%
Episode  3500 | epsilon=0.307 | success rate (last 500 eps) = 60.00%
Episode  4000 | epsilon=0.208 | success rate (last 500 eps) = 69.80%
Episode  4500 | epsilon=0.109 | success rate (last 500 eps) = 78.60%
Episode  5000 | epsilon=0.010 | success rate (last 500 eps) = 93.80%

Final Q-table:
State |    LEFT |    DOWN |   RIGHT |      UP
    0 |   0.941 |   0.951 |   0.951 |   0.941
    1 |   0.941 |   0.000 |   0.961 |   0.951
    2 |   0.951 |   0.970 |   0.951 |   0.961
    3 |   0.961 |   0.000 |   0.95